In [ ]:
%cd /content
!rm -rf /content/DVARF
!git clone https://github.com/AntoFratta/DVARF.git

%cd /content/DVARF
!grep -v "triton-windows" requirements.txt > requirements_colab.txt
!pip install -r requirements_colab.txt

In [ ]:
%cd /content
!rm -rf /content/sam3
!git clone https://github.com/facebookresearch/sam3.git

%cd /content/sam3
!pip -q install -e .

In [ ]:
from huggingface_hub import login
login()

In [ ]:
import numpy as np
import torch
from sam3.model_builder import build_sam3_image_model

print("numpy:", np.__version__)
device = "cuda" if torch.cuda.is_available() else "cpu"
_ = build_sam3_image_model(device=device, load_from_HF=True)
print("SAM3 build OK | device:", device)

In [ ]:
import os, sys
from pathlib import Path

os.environ["SAM3_DEBUG"] = "0" # evita spam di log

PROJECT_ROOT = Path("/content/DVARF")
if str(PROJECT_ROOT) not in sys.path:
  sys.path.insert(0, str(PROJECT_ROOT))

import src.sam3_wrapper as sw
print("Wrapper importato da:", sw.__file__)

In [ ]:
# CELL 1 — setup paths + loader FIX per dataclasses (Python 3.12)
import os, sys
from pathlib import Path
import importlib.util

PROJECT_ROOT = Path("/content/DVARF")
SCRIPTS_DIR = PROJECT_ROOT / "scripts"

os.environ["SAM3_DEBUG"] = "0"

def load_py(name: str, path: Path):
    """Carica un .py come modulo (fix per dataclasses su Py3.12)."""
    spec = importlib.util.spec_from_file_location(name, str(path))
    mod = importlib.util.module_from_spec(spec)
    assert spec and spec.loader
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

# carico i tuoi script come moduli
run_mod   = load_py("run_sam3_on_split", SCRIPTS_DIR / "run_sam3_on_split.py")
build_mod = load_py("build_linear_probe_dataset", SCRIPTS_DIR / "build_linear_probe_dataset.py")
train_mod = load_py("train_linear_probe", SCRIPTS_DIR / "train_linear_probe.py")
apply_mod = load_py("apply_linear_probe_to_split", SCRIPTS_DIR / "apply_linear_probe_to_split.py")

print("OK: script caricati.")
print(" -", run_mod.__file__)
print(" -", build_mod.__file__)
print(" -", train_mod.__file__)
print(" -", apply_mod.__file__)



In [ ]:
# CELL 2 — Check dataset (serve per evitare run inutili)
RAW_IMG = PROJECT_ROOT / "data" / "raw" / "images"
RAW_LBL = PROJECT_ROOT / "data" / "raw" / "labels"

for split in ["train", "val", "test"]:
    img_dir = RAW_IMG / split
    lbl_dir = RAW_LBL / split
    imgs = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png"))
    lbls = list(lbl_dir.glob("*.txt"))

    print(f"\n[{split}] images={len(imgs)} labels={len(lbls)}")
    if not img_dir.exists() or not lbl_dir.exists():
        raise FileNotFoundError(f"Mancano le cartelle per split={split}: {img_dir} / {lbl_dir}")
    if len(imgs) == 0 or len(lbls) == 0:
        raise RuntimeError(f"Split={split} vuoto (immagini o labels mancanti).")



In [ ]:
# CELL 3 — Leggi le soglie “ufficiali” dal progetto (config.py)
from src.config import EXPORT_THRESHOLD_DEFAULT, EVAL_THRESHOLD_DEFAULT, NMS_IOU_DEFAULT, NMS_MAX_DET_DEFAULT

print("EXPORT_THRESHOLD_DEFAULT:", EXPORT_THRESHOLD_DEFAULT, "(soglia bassa per salvare candidati)")
print("EVAL_THRESHOLD_DEFAULT  :", EVAL_THRESHOLD_DEFAULT, "(placeholder; quella 'giusta' la scegliamo con sweep su VAL)")
print("NMS_IOU_DEFAULT         :", NMS_IOU_DEFAULT)
print("NMS_MAX_DET_DEFAULT     :", NMS_MAX_DET_DEFAULT)



In [ ]:
# CELL 4 — pulizia output precedenti
import shutil

paths_to_clean = [
    PROJECT_ROOT / "data" / "processed" / "predictions" / "sam3_yolo",
    PROJECT_ROOT / "data" / "processed" / "predictions" / "sam3_linear_probe_yolo",
    PROJECT_ROOT / "data" / "processed" / "features" / "sam3_prehead",
    PROJECT_ROOT / "data" / "processed" / "linear_probe",
    PROJECT_ROOT / "results" / "threshold_sweeps",
]

for p in paths_to_clean:
    if p.exists():
        shutil.rmtree(p)
        print("Removed:", p)
    else:
        print("Skip:", p)


In [ ]:
# CELL 5 — Run SAM3 su train/val/test usando export_threshold dal config
import inspect

def run_sam3_split(split: str):
    sig = inspect.signature(run_mod.run_sam3_on_split)
    kwargs = dict(
        split=split,
        max_images=None,                   # FULL
        save_segmentations=True,
        max_masks_per_image_per_class=None,
        nms_iou=NMS_IOU_DEFAULT,
        nms_max_det=NMS_MAX_DET_DEFAULT,
    )
    # compatibilità nome parametro soglia
    if "export_threshold" in sig.parameters:
        kwargs["export_threshold"] = EXPORT_THRESHOLD_DEFAULT
    else:
        kwargs["score_threshold"] = EXPORT_THRESHOLD_DEFAULT

    print("\n" + "="*90)
    print(f"RUN SAM3 | split={split} | export_thr={EXPORT_THRESHOLD_DEFAULT} | segm=True")
    print("="*90)
    run_mod.run_sam3_on_split(**kwargs)

for split in ["train", "val", "test"]:
    run_sam3_split(split)



In [ ]:
# CELL 6 — Sweep su VAL per scegliere automaticamente la soglia “giusta” (baseline)
import numpy as np
import pandas as pd

from src.eval_yolo import evaluate_yolo_predictions
from src.config import get_labels_dir, get_sam3_yolo_predictions_dir
from src.prompts import CLASS_PROMPTS

def sweep_thresholds_for_dir(labels_dir, preds_dir, thresholds):
    rows = []
    for thr in thresholds:
        res = evaluate_yolo_predictions(
            labels_dir=labels_dir,
            preds_dir=preds_dir,
            num_classes=len(CLASS_PROMPTS),
            confidence_threshold=float(thr),
        )
        rows.append({
            "thr": float(thr),
            "micro_P": res.micro_precision,
            "micro_R": res.micro_recall,
            "micro_F1": res.micro_f1,
            "mAP50": res.map_50,
            "mAP50_95": res.map_50_95,
            "total_pred": res.total_pred,
        })
    df = pd.DataFrame(rows)
    best_f1 = df.loc[df["micro_F1"].idxmax()].to_dict()
    return df, best_f1

thresholds = np.round(np.linspace(0.05, 0.95, 19), 2)

labels_val = get_labels_dir("val")
preds_val_base = get_sam3_yolo_predictions_dir("val")

df_base, best_base = sweep_thresholds_for_dir(labels_val, preds_val_base, thresholds)
THR_BASELINE = best_base["thr"]

print("BEST baseline (VAL) by micro-F1:", best_base)
display(df_base.sort_values("micro_F1", ascending=False).head(10))



In [ ]:
# CELL 7 — Eval baseline su TEST usando la soglia scelta su VAL
from src.eval_yolo import print_evaluation_summary
from src.prompts import CLASS_PROMPTS

labels_test = get_labels_dir("test")
preds_test_base = get_sam3_yolo_predictions_dir("test")

res_base_test = evaluate_yolo_predictions(
    labels_dir=labels_test,
    preds_dir=preds_test_base,
    num_classes=len(CLASS_PROMPTS),
    confidence_threshold=float(THR_BASELINE),
)

print("\n" + "="*90)
print(f"BASELINE TEST | eval_thr (from VAL best-F1) = {THR_BASELINE}")
print("="*90)

class_names = {cid: name for cid, name in CLASS_PROMPTS.items()}
print_evaluation_summary(res_base_test, class_names=class_names)


In [ ]:
# CELL 8 — Build dataset probe su TRAIN (IoU matching = 0.5 come nel progetto)
print("\n" + "="*90)
print("BUILD LINEAR PROBE DATASET | train")
print("="*90)

dataset_path = build_mod.build_linear_probe_dataset_for_split(
    split="train",
    confidence_threshold=float(EXPORT_THRESHOLD_DEFAULT),  # stessi candidati esportati
    iou_threshold=0.5,  # matching IoU standard
)
print("Dataset salvato in:", dataset_path)




In [ ]:
# CELL 9 — Train linear probe su TRAIN
print("\n" + "="*90)
print("TRAIN LINEAR PROBE | train")
print("="*90)

weights_path = train_mod.train_linear_probe(
    split="train",
    num_epochs=500,
    learning_rate=0.1,
    l2_weight=1e-4,
)
print("Pesi salvati in:", weights_path)


In [ ]:
import numpy as np
from pathlib import Path

weights_path = Path(PROJECT_ROOT) / "data" / "processed" / "linear_probe" / "sam3_linear_probe_weights.npz"
w = np.load(weights_path)

print("Keys:", w.files)
print("weights shape:", w["weights"].shape, "biases shape:", w["biases"].shape)

has_bbox = ("bbox_weights" in w.files) and ("bbox_biases" in w.files)
print("Has bbox refinement:", has_bbox)
if has_bbox:
    print("bbox_weights shape:", w["bbox_weights"].shape, "bbox_biases shape:", w["bbox_biases"].shape)


In [ ]:
#CELL 10 — Apply probe su VAL e TEST (rigenera i .txt con score ricalcolati)
for split in ["val", "test"]:
    print("\n" + "="*90)
    print(f"APPLY PROBE | split={split}")
    print("="*90)
    out_dir = apply_mod.apply_linear_probe_to_split(split=split)
    print("Output:", out_dir)



In [ ]:
from pathlib import Path
import numpy as np

pred_base = Path(PROJECT_ROOT) / "data/processed/predictions/sam3_yolo/val"
pred_probe = Path(PROJECT_ROOT) / "data/processed/predictions/sam3_linear_probe_yolo/val"

files = sorted(pred_base.glob("*.txt"))[:20]

def load_preds(p):
    rows=[]
    with open(p, "r", encoding="utf-8") as f:
        for ln in f:
            parts = ln.strip().split()
            if len(parts) < 5:
                continue
            cid = int(parts[0])
            cx,cy,w,h = map(float, parts[1:5])
            score = float(parts[5]) if len(parts) >= 6 else 1.0
            rows.append((cid,cx,cy,w,h,score))
    return rows

deltas=[]
for f in files:
    fb = pred_base / f.name
    fp = pred_probe / f.name
    if not fp.exists():
        continue
    B = load_preds(fb)
    P = load_preds(fp)
    # confronto "grezzo": prendo i primi min(len) (non è matching perfetto, serve solo come sanity)
    m = min(len(B), len(P))
    for i in range(m):
        deltas.append([
            abs(P[i][1]-B[i][1]),
            abs(P[i][2]-B[i][2]),
            abs(P[i][3]-B[i][3]),
            abs(P[i][4]-B[i][4]),
        ])

deltas = np.array(deltas) if deltas else None
print("Num compared boxes:", 0 if deltas is None else len(deltas))
if deltas is not None and len(deltas)>0:
    print("Mean |Δcx,Δcy,Δw,Δh|:", deltas.mean(axis=0))
    print("Max  |Δcx,Δcy,Δw,Δh|:", deltas.max(axis=0))


In [ ]:
import numpy as np
from pathlib import Path

labels_val = Path(PROJECT_ROOT) / "data/raw/labels/val"
preds_base_val = Path(PROJECT_ROOT) / "data/processed/predictions/sam3_yolo/val"
preds_probe_val = Path(PROJECT_ROOT) / "data/processed/predictions/sam3_linear_probe_yolo/val"

def yolo_to_xyxy(cx, cy, w, h):
    x1 = cx - w/2
    y1 = cy - h/2
    x2 = cx + w/2
    y2 = cy + h/2
    return np.array([x1,y1,x2,y2], dtype=np.float32)

def iou(box, boxes):
    if boxes.size == 0:
        return np.zeros((0,), dtype=np.float32)
    box = box.reshape(1,4)
    x1 = np.maximum(box[:,0], boxes[:,0])
    y1 = np.maximum(box[:,1], boxes[:,1])
    x2 = np.minimum(box[:,2], boxes[:,2])
    y2 = np.minimum(box[:,3], boxes[:,3])
    inter = np.clip(x2-x1, 0, None) * np.clip(y2-y1, 0, None)
    area1 = (box[:,2]-box[:,0])*(box[:,3]-box[:,1])
    area2 = (boxes[:,2]-boxes[:,0])*(boxes[:,3]-boxes[:,1])
    union = area1 + area2 - inter
    return (inter / np.clip(union, 1e-9, None)).astype(np.float32)

def load_labels(path):
    gt=[]
    if not path.exists():
        return gt
    for ln in path.read_text().splitlines():
        p = ln.strip().split()
        if len(p) < 5:
            continue
        cid = int(p[0])
        cx,cy,w,h = map(float, p[1:5])
        gt.append((cid, cx,cy,w,h))
    return gt

def load_preds(path, thr=0.0):
    pr=[]
    if not path.exists():
        return pr
    for ln in path.read_text().splitlines():
        p = ln.strip().split()
        if len(p) < 5:
            continue
        cid = int(p[0])
        cx,cy,w,h = map(float, p[1:5])
        score = float(p[5]) if len(p) >= 6 else 1.0
        if score >= thr:
            pr.append((cid, cx,cy,w,h,score))
    return pr

# usa una soglia bassa qui per vedere la "pura" qualità della localizzazione.
# (poi puoi rifarla anche a eval_threshold=0.26)
IOU_DEBUG_SCORE_THR = 0.05

deltas=[]
improved=0
worsened=0
total=0

label_files = sorted(labels_val.glob("*.txt"))

for lf in label_files:
    name = lf.name
    gt = load_labels(lf)
    pb = load_preds(preds_base_val / name, thr=IOU_DEBUG_SCORE_THR)
    pp = load_preds(preds_probe_val / name, thr=IOU_DEBUG_SCORE_THR)

    # indicizza preds per classe
    byc_b = {}
    byc_p = {}
    for cid,cx,cy,w,h,s in pb:
        byc_b.setdefault(cid, []).append(yolo_to_xyxy(cx,cy,w,h))
    for cid,cx,cy,w,h,s in pp:
        byc_p.setdefault(cid, []).append(yolo_to_xyxy(cx,cy,w,h))

    for cid,cx,cy,w,h in gt:
        g = yolo_to_xyxy(cx,cy,w,h)
        Bb = np.stack(byc_b.get(cid, []), axis=0) if cid in byc_b else np.zeros((0,4), np.float32)
        Bp = np.stack(byc_p.get(cid, []), axis=0) if cid in byc_p else np.zeros((0,4), np.float32)

        iou_b = float(iou(g, Bb).max()) if Bb.shape[0] else 0.0
        iou_p = float(iou(g, Bp).max()) if Bp.shape[0] else 0.0

        d = iou_p - iou_b
        deltas.append(d)
        total += 1
        if d > 1e-3: improved += 1
        elif d < -1e-3: worsened += 1

deltas = np.array(deltas, dtype=np.float32)
print("GT boxes evaluated:", total)
print("Mean ΔIoU:", float(deltas.mean()))
print("Median ΔIoU:", float(np.median(deltas)))
print("% improved:", 100.0*improved/max(total,1), "% worsened:", 100.0*worsened/max(total,1))
print("ΔIoU percentiles:", np.percentile(deltas, [5,25,50,75,95]))


In [ ]:
#CELL 11 — Sweep su VAL per scegliere la soglia “giusta” del PROBE
from src.config import PREDICTIONS_DIR

preds_val_probe = PREDICTIONS_DIR / "sam3_linear_probe_yolo" / "val"

df_probe, best_probe = sweep_thresholds_for_dir(labels_val, preds_val_probe, thresholds)
THR_PROBE = best_probe["thr"]

print("BEST probe (VAL) by micro-F1:", best_probe)
display(df_probe.sort_values("micro_F1", ascending=False).head(10))


In [ ]:
#CELL 12 — Eval probe su TEST usando la soglia scelta su VAL

preds_test_probe = PREDICTIONS_DIR / "sam3_linear_probe_yolo" / "test"

res_probe_test = evaluate_yolo_predictions(
    labels_dir=labels_test,
    preds_dir=preds_test_probe,
    num_classes=len(CLASS_PROMPTS),
    confidence_threshold=float(THR_PROBE),
)

print("\n" + "="*90)
print(f"PROBE TEST | eval_thr (from VAL best-F1) = {THR_PROBE}")
print("="*90)
print_evaluation_summary(res_probe_test, class_names=class_names)


In [ ]:
#CELL 13 — (Consigliata) salva un report txt con soglie e confronto finale

report_path = PROJECT_ROOT / "results" / "final_report.txt"
report_path.parent.mkdir(parents=True, exist_ok=True)

with open(report_path, "w", encoding="utf-8") as f:
    f.write("=== THRESHOLD SYSTEM ===\n")
    f.write(f"Export threshold (candidate generation): {EXPORT_THRESHOLD_DEFAULT}\n")
    f.write(f"Baseline eval threshold (chosen on VAL by best micro-F1): {THR_BASELINE}\n")
    f.write(f"Probe eval threshold (chosen on VAL by best micro-F1): {THR_PROBE}\n\n")

    f.write("=== NOTE ===\n")
    f.write("- Export threshold bassa per non scartare candidati utili al probe.\n")
    f.write("- Eval threshold selezionata su validation per definire l’operating point.\n\n")

    f.write("=== BASELINE TEST SUMMARY ===\n")
    f.write(f"micro_P={res_base_test.micro_precision:.4f} micro_R={res_base_test.micro_recall:.4f} micro_F1={res_base_test.micro_f1:.4f}\n")
    f.write(f"mAP50={res_base_test.map_50:.4f} mAP50_95={res_base_test.map_50_95:.4f}\n\n")

    f.write("=== PROBE TEST SUMMARY ===\n")
    f.write(f"micro_P={res_probe_test.micro_precision:.4f} micro_R={res_probe_test.micro_recall:.4f} micro_F1={res_probe_test.micro_f1:.4f}\n")
    f.write(f"mAP50={res_probe_test.map_50:.4f} mAP50_95={res_probe_test.map_50_95:.4f}\n")

print("Report salvato in:", report_path)
print("Puoi scaricarlo da Files di Colab:", report_path)


In [ ]:
# Confronto FAIR: stessa soglia su TEST per baseline e probe
from src.eval_yolo import evaluate_yolo_predictions, print_evaluation_summary
from src.config import get_labels_dir, get_sam3_yolo_predictions_dir, PREDICTIONS_DIR
from src.prompts import CLASS_PROMPTS

THR_FAIR = 0.35  # usa la stessa soglia per entrambi (es. quella best-F1 del probe)

labels_test = get_labels_dir("test")
preds_test_base  = get_sam3_yolo_predictions_dir("test")
preds_test_probe = PREDICTIONS_DIR / "sam3_linear_probe_yolo" / "test"

class_names = {cid: name for cid, name in CLASS_PROMPTS.items()}

def eval_dir(name, preds_dir):
    res = evaluate_yolo_predictions(
        labels_dir=labels_test,
        preds_dir=preds_dir,
        num_classes=len(CLASS_PROMPTS),
        confidence_threshold=float(THR_FAIR),
    )
    print("\n" + "="*90)
    print(f"{name} TEST | eval_thr = {THR_FAIR}")
    print("="*90)
    print_evaluation_summary(res, class_names=class_names)
    return res

res_base = eval_dir("BASELINE", preds_test_base)
res_probe = eval_dir("PROBE", preds_test_probe)

print("\n" + "="*90)
print("DELTA (PROBE - BASELINE) @ same threshold")
print("="*90)
print(f"micro_P:  {res_probe.micro_precision - res_base.micro_precision:+.4f}")
print(f"micro_R:  {res_probe.micro_recall    - res_base.micro_recall:+.4f}")
print(f"micro_F1: {res_probe.micro_f1        - res_base.micro_f1:+.4f}")
print(f"mAP50:    {res_probe.map_50          - res_base.map_50:+.4f}")
print(f"mAP50_95: {res_probe.map_50_95       - res_base.map_50_95:+.4f}")


In [ ]:
# === Mean IoU (bbox-level) on TEST: matched TPs only (IoU>=0.5) ===

from pathlib import Path
from PIL import Image
import numpy as np

PROJECT_ROOT = Path("/content/DVARF")
IMG_DIR = PROJECT_ROOT / "data/raw/images/test"
LBL_DIR = PROJECT_ROOT / "data/raw/labels/test"

PRED_BASE_DIR  = PROJECT_ROOT / "data/processed/predictions/sam3_yolo/test"
PRED_PROBE_DIR = PROJECT_ROOT / "data/processed/predictions/sam3_linear_probe_yolo/test"

CLASS_NAMES = {0: "crashed car", 1: "person", 2: "car"}

def yolo_to_xyxy(box, W, H):
    xc, yc, w, h = box
    x1 = (xc - w/2) * W
    y1 = (yc - h/2) * H
    x2 = (xc + w/2) * W
    y2 = (yc + h/2) * H
    return np.array([x1, y1, x2, y2], dtype=np.float32)

def iou_xyxy(a, b):
    ix1 = max(a[0], b[0]); iy1 = max(a[1], b[1])
    ix2 = min(a[2], b[2]); iy2 = min(a[3], b[3])
    iw = max(0.0, ix2 - ix1); ih = max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, a[2]-a[0]) * max(0.0, a[3]-a[1])
    area_b = max(0.0, b[2]-b[0]) * max(0.0, b[3]-b[1])
    union = area_a + area_b - inter
    return 0.0 if union <= 0 else float(inter / union)

def read_gt_boxes(lbl_path):
    out = []
    if not lbl_path.exists():
        return out
    txt = lbl_path.read_text().strip()
    if not txt:
        return out
    for line in txt.splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        c = int(parts[0])
        xc, yc, w, h = map(float, parts[1:5])
        out.append((c, (xc, yc, w, h)))
    return out

def read_pred_boxes(pred_path, score_thr):
    out = []
    if not pred_path.exists():
        return out
    txt = pred_path.read_text().strip()
    if not txt:
        return out
    for line in txt.splitlines():
        parts = line.split()
        if len(parts) < 6:
            continue
        c = int(float(parts[0]))
        s = float(parts[1])
        if s < score_thr:
            continue
        xc, yc, w, h = map(float, parts[2:6])
        out.append((c, s, (xc, yc, w, h)))
    return out

def match_and_collect_ious(img_path, gt, preds, match_iou_thr=0.5):
    W, H = Image.open(img_path).size
    ious_by_class = {0: [], 1: [], 2: []}

    for cls in [0, 1, 2]:
        gt_cls = [yolo_to_xyxy(b, W, H) for (c, b) in gt if c == cls]
        pr_cls = [(s, yolo_to_xyxy(b, W, H)) for (c, s, b) in preds if c == cls]

        if len(gt_cls) == 0 or len(pr_cls) == 0:
            continue

        pr_cls.sort(key=lambda x: x[0], reverse=True)

        matched_gt = set()
        for s, pb in pr_cls:
            best_iou = 0.0
            best_j = None
            for j, gb in enumerate(gt_cls):
                if j in matched_gt:
                    continue
                cur = iou_xyxy(pb, gb)
                if cur > best_iou:
                    best_iou = cur
                    best_j = j
            if best_j is not None and best_iou >= match_iou_thr:
                matched_gt.add(best_j)
                ious_by_class[cls].append(best_iou)

    return ious_by_class

def mean_iou_for_dir(pred_dir, score_thr, match_iou_thr=0.5):
    all_ious = {0: [], 1: [], 2: []}
    img_files = sorted(list(IMG_DIR.glob("*.jpg")) + list(IMG_DIR.glob("*.png")))

    for img_path in img_files:
        stem = img_path.stem
        gt = read_gt_boxes(LBL_DIR / f"{stem}.txt")
        preds = read_pred_boxes(pred_dir / f"{stem}.txt", score_thr=score_thr)
        ious = match_and_collect_ious(img_path, gt, preds, match_iou_thr=match_iou_thr)
        for k in all_ious:
            all_ious[k].extend(ious[k])

    per_class = {k: (float(np.mean(v)) if len(v) > 0 else float("nan")) for k, v in all_ious.items()}
    all_tp_ious = [x for k in all_ious for x in all_ious[k]]
    global_mean = float(np.mean(all_tp_ious)) if len(all_tp_ious) > 0 else float("nan")
    counts = {k: len(all_ious[k]) for k in all_ious}
    return per_class, global_mean, counts

# Se nel notebook esistono già THR_BASELINE e THR_PROBE, usiamo quelli:
print("THR_BASELINE (da VAL):", THR_BASELINE)
print("THR_PROBE    (da VAL):", THR_PROBE)

# 1) Standard: soglie ottimizzate su VAL (tabella B)
base_pc, base_m, base_cnt = mean_iou_for_dir(PRED_BASE_DIR,  score_thr=float(THR_BASELINE), match_iou_thr=0.5)
prob_pc, prob_m, prob_cnt = mean_iou_for_dir(PRED_PROBE_DIR, score_thr=float(THR_PROBE),    match_iou_thr=0.5)

print("\n=== Mean IoU (matched boxes) | STANDARD (VAL-tuned thresholds) ===")
print(f"Baseline @thr={THR_BASELINE}")
for c in [0,1,2]:
    print(f"  {CLASS_NAMES[c]:12s}: meanIoU={base_pc[c]:.4f} | TP matches={base_cnt[c]}")
print(f"  Global mean IoU (matched boxes): {base_m:.4f}")

print(f"\nProbe @thr={THR_PROBE}")
for c in [0,1,2]:
    print(f"  {CLASS_NAMES[c]:12s}: meanIoU={prob_pc[c]:.4f} | TP matches={prob_cnt[c]}")
print(f"  Global mean IoU (matched boxes): {prob_m:.4f}")

# 2) Fair: stessa soglia per entrambi (se vuoi anche questa riga in tabella)
THR_FAIR = 0.35
base_pc_f, base_m_f, _ = mean_iou_for_dir(PRED_BASE_DIR,  score_thr=float(THR_FAIR), match_iou_thr=0.5)
prob_pc_f, prob_m_f, _ = mean_iou_for_dir(PRED_PROBE_DIR, score_thr=float(THR_FAIR), match_iou_thr=0.5)

print("\n=== Mean IoU (matched boxes) | FAIR (same threshold) ===")
print(f"Both @thr={THR_FAIR}")
print(f"  Baseline global mean IoU: {base_m_f:.4f}")
print(f"  Probe    global mean IoU: {prob_m_f:.4f}")


In [ ]:
# === Speed (ms/frame) on TEST ===

import time
import inspect

N_TEST = 100  # il tuo test set ha 100 immagini

def timed_run_sam3_test():
    sig = inspect.signature(run_mod.run_sam3_on_split)
    kwargs = dict(
        split="test",
        max_images=None,
        save_segmentations=False,  # speed realistico per pipeline detection
        max_masks_per_image_per_class=None,
        nms_iou=NMS_IOU_DEFAULT,
        nms_max_det=NMS_MAX_DET_DEFAULT,
    )
    if "export_threshold" in sig.parameters:
        kwargs["export_threshold"] = EXPORT_THRESHOLD_DEFAULT
    else:
        kwargs["score_threshold"] = EXPORT_THRESHOLD_DEFAULT

    t0 = time.perf_counter()
    run_mod.run_sam3_on_split(**kwargs)
    t1 = time.perf_counter()
    return (t1 - t0)

def timed_apply_probe_test():
    t0 = time.perf_counter()
    apply_mod.apply_linear_probe_to_split(split="test")
    t1 = time.perf_counter()
    return (t1 - t0)

sec_base = timed_run_sam3_test()
ms_frame_base = (sec_base / N_TEST) * 1000.0
print(f"\nBaseline SAM3 | total={sec_base:.2f}s | speed={ms_frame_base:.2f} ms/frame")

sec_apply = timed_apply_probe_test()
ms_frame_apply = (sec_apply / N_TEST) * 1000.0
print(f"Apply Probe    | total={sec_apply:.2f}s | speed={ms_frame_apply:.2f} ms/frame")

print(f"Total pipeline | speed={(ms_frame_base + ms_frame_apply):.2f} ms/frame (SAM3 + probe)")


In [ ]:
from pathlib import Path
from datetime import datetime

out_path = Path("/content/DVARF/results/extra_metrics_miou_speed.txt")
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w", encoding="utf-8") as f:
    f.write("EXTRA METRICS (notebook)\n")
    f.write(f"Timestamp: {datetime.now().isoformat()}\n\n")

    f.write("Mean IoU (matched bounding boxes, IoU>=0.5)\n")
    f.write(f"  Baseline (thr={THR_BASELINE}): global_mean_iou={base_m:.4f}\n")
    f.write(f"  Probe    (thr={THR_PROBE}): global_mean_iou={prob_m:.4f}\n\n")

    f.write("Speed (ms/frame) on TEST\n")
    f.write(f"  Baseline SAM3: {ms_frame_base:.2f} ms/frame\n")
    f.write(f"  Apply probe  : {ms_frame_apply:.2f} ms/frame\n")
    f.write(f"  Total pipeline (SAM3+probe): {(ms_frame_base + ms_frame_apply):.2f} ms/frame\n")

print("Salvato file:", out_path)


In [ ]:
%cd /content/DVARF
!zip -r processed.zip data/processed
!zip -r results.zip results
